# Prédiction de la consommation électrique — RNN/LSTM/GRU + MLP hybride (population complète)

## Objectif

Ce notebook reprend l'architecture hybride de `rnn_mlp_hybrid_electricite.ipynb` (branche
séquentielle RNN/LSTM/GRU + branche statique MLP, concaténées puis passées dans une tête de
régression commune), mais l'applique à **tous les bâtiments qui passent le filtre**, au lieu des
100 bâtiments tirés aléatoirement parmi les candidats.

Le filtre est celui déjà défini dans `timeseries_clustering_multi.ipynb` : tout-électrique
strict + `Single-Family Detached` + 1 étage (**plain-pied**) + pas de véhicule électrique / piscine
/ panneaux solaires + climatisation centrale + chauffage électrique par gaines + zones climatiques
tempérées (3A/4A/5A). Sur les 549 971 logements du dataset complet, ce filtre retient **6 704
bâtiments candidats** (contre 100 dans le notebook d'origine).

## Ce qui change par rapport au notebook à 100 bâtiments

- **Partie A** (prévision t+1, fenêtre de 24h) : à 100 bâtiments, les fenêtres glissantes de
  l'ensemble d'entraînement (~611k fenêtres) tenaient en RAM et étaient matérialisées une fois
  pour toutes. À 6 704 bâtiments, l'ensemble d'entraînement représenterait plus de 40 millions de
  fenêtres (~30 Go rien que pour la branche séquentielle) — impossible à matérialiser sur cette
  machine (12 Go de RAM). La Partie A utilise donc un générateur (`RandomWindowSequence`) qui tire
  un sous-échantillon aléatoire de fenêtres à la volée directement depuis les séries horaires
  brutes tenues en RAM (`hourly_all`, ~2 Go pour 6 704 bâtiments), renouvelé à chaque epoch pour
  l'entraînement et fixé pour la validation/le test.
- **Partie B** (extrapolation annuelle) : une seule observation par bâtiment, donc pas de problème
  de mémoire à cette échelle — mais l'échantillon d'entraînement passe de 70 à plusieurs milliers
  de bâtiments, ce qui règle directement la limite principale identifiée dans le notebook
  d'origine («samples too small pour un réseau de neurones»). La capacité du réseau est donc
  légèrement augmentée par rapport à la version à 100 bâtiments (elle n'a plus besoin d'être bridée
  pour éviter le sur-apprentissage sur 70 exemples).

## Volume de données et temps d'exécution

Télécharger les séries temporelles des ~6 700 bâtiments représente environ **65 Go** (contre 1 Go
pour l'échantillon à 100 bâtiments) et peut prendre plusieurs heures selon la bande passante. Le
téléchargement (cellule dédiée plus bas) est **idempotent et reprenable** : il ne retélécharge pas
les fichiers déjà présents, donc ce notebook peut être relancé/interrompu sans perdre de travail.
Le reste du pipeline (chargement des séries, entraînement) tolère aussi un sous-ensemble partiel
de bâtiments déjà téléchargés — pas besoin d'attendre les 6 704 fichiers pour commencer à itérer.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Bride TensorFlow (16 cœurs dispos par défaut) pour éviter la surchauffe/freeze pendant l'entraînement CPU
tf.config.threading.set_intra_op_parallelism_threads(6)
tf.config.threading.set_inter_op_parallelism_threads(2)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

ROOT = Path().resolve().parent.parent
DATA_RAW = ROOT / "data" / "raw"
DATA_RAW_FULL = DATA_RAW / "timeseries_full_all_electric"
DATA_PROCESSED = ROOT / "data" / "processed"
FIGURES = ROOT / "reports" / "figures"

COL = "out.electricity.total.energy_consumption..kwh"
WEATHER_COLS = [
    "out.outdoor_air_drybulb_temp..c",
    "out.outdoor_air_relative_humidity..percentage",
    "out.outdoor_air_wetbulb_temp..c",
    "out.outdoor_humidity_ratio..kgwater_per_kgdryair",
    "out.weather.diffuse_solar_radiation..watt_per_m2",
    "out.weather.direct_normal_solar_radiation..watt_per_m2",
    "out.weather.wind_speed..meter_per_second",
]
SEQ_COLS = [COL] + WEATHER_COLS
N_SEQ_FEATURES = len(SEQ_COLS)
N_HOURS = 8760

# Palette/styles daltonisme (mêmes conventions que dans timeseries_clustering_multi.ipynb --
# ordre fixe, ne jamais permuter)
CB_PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
CB_HATCHES = ["", "//", "xx", "\\\\", "..", "++", "oo", "**"]

## Population complète des bâtiments filtrés

Même filtre que `timeseries_clustering_multi.ipynb` (voir en tête de notebook), mais sans tirage
aléatoire à 100 bâtiments : on garde ici **tous** les candidats. La liste est sauvegardée
séparément (`buildings_all_electric_full.parquet`) pour ne pas toucher au fichier
`buildings_all_electric_100.parquet` utilisé par le notebook à 100 bâtiments.

In [ ]:
BUILDINGS_LIST_PATH = DATA_PROCESSED / "buildings_all_electric_full.parquet"

raw_meta = pd.read_parquet(DATA_RAW / "upgrade0.parquet", columns=[
    "bldg_id", "in.state", "in.geometry_building_type_recs", "in.geometry_stories",
    "in.electric_vehicle_ownership", "in.misc_pool", "in.has_pv",
    "in.hvac_cooling_type", "in.hvac_heating_type", "in.heating_fuel",
    "in.ashrae_iecc_climate_zone_2004",
    "out.natural_gas.total.energy_consumption..kwh",
    "out.fuel_oil.total.energy_consumption..kwh",
    "out.propane.total.energy_consumption..kwh",
])

fuel_totals = [
    "out.natural_gas.total.energy_consumption..kwh",
    "out.fuel_oil.total.energy_consumption..kwh",
    "out.propane.total.energy_consumption..kwh",
]
all_electric_mask = (raw_meta[fuel_totals].fillna(0) == 0).all(axis=1)

# Filtre identique à timeseries_clustering_multi.ipynb (bâtiments individuels, 1 étage =
# plain-pied, sans EV/piscine/PV, AC centrale + chauffage électrique par gaines, zones climat
# tempérées 3A/4A/5A) -- population complète, pas d'échantillonnage à 100 ici.
filter_mask = (
    (raw_meta["in.geometry_building_type_recs"] == "Single-Family Detached") &
    (raw_meta["in.geometry_stories"] == "1") &
    (raw_meta["in.electric_vehicle_ownership"] == "No") &
    (raw_meta["in.misc_pool"] == "None") &
    (raw_meta["in.has_pv"] == "No") &
    (raw_meta["in.hvac_cooling_type"] == "Central AC") &
    (raw_meta["in.hvac_heating_type"] == "Ducted Heating") &
    (raw_meta["in.heating_fuel"] == "Electricity") &
    (raw_meta["in.ashrae_iecc_climate_zone_2004"].isin(["3A", "4A", "5A"])) &
    all_electric_mask
)

candidates = raw_meta.loc[filter_mask, ["bldg_id", "in.state", "in.ashrae_iecc_climate_zone_2004"]].reset_index(drop=True)
print(f"{len(candidates):,} bâtiments candidats (filtre tout-électrique + plain-pied, population complète)")

if BUILDINGS_LIST_PATH.exists():
    buildings_full = pd.read_parquet(BUILDINGS_LIST_PATH)
else:
    buildings_full = candidates.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    buildings_full.to_parquet(BUILDINGS_LIST_PATH, index=False)

print(f"Bâtiments retenus : {len(buildings_full):,}")
buildings_full.head()

## Téléchargement des séries temporelles (~65 Go, idempotent, parallélisé)

Mesuré en conditions réelles sur ce réseau : chaque fichier ne pèse que ~6,5 Mo, mais un
téléchargement séquentiel (un fichier après l'autre, comme dans `timeseries_clustering_multi.ipynb`)
passe le plus clair de son temps à rouvrir une connexion pour chaque fichier (~3,4 s/fichier tout
compris) plutôt qu'à transférer des données — soit environ **6h** pour 6 704 fichiers. Le goulot
d'étranglement est la latence par requête, pas la bande passante : `ThreadPoolExecutor` lance
`N_WORKERS` téléchargements en même temps (leurs temps de connexion se chevauchent au lieu de
s'additionner), ce qui ramène l'estimation à environ **35-45 minutes**.

Comme dans `timeseries_clustering_multi.ipynb`, la cellule ne télécharge que les fichiers
manquants dans `data/raw/timeseries_full_all_electric/` : elle peut être interrompue et relancée
sans perdre les fichiers déjà récupérés. Les échecs (ex. bâtiment absent du bucket) sont consignés
dans `buildings_full_download_failed.parquet` plutôt que de faire échouer tout le run.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

OEDI_BASE = (
    "https://oedi-data-lake.s3.amazonaws.com/"
    "nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/"
    "2025/resstock_amy2018_release_1/"
    "timeseries_individual_buildings/by_state/upgrade=0"
)

DATA_RAW_FULL.mkdir(parents=True, exist_ok=True)
N_WORKERS = 10  # téléchargements simultanés -- au-delà, risque de limitation côté S3


def download_one(bid, state):
    out = DATA_RAW_FULL / f"{bid}-0.parquet"
    if out.exists():
        return bid, True, None
    url = f"{OEDI_BASE}/state={state}/{bid}-0.parquet"
    try:
        df_ts = pd.read_parquet(url)
        df_ts.to_parquet(out)
        return bid, True, None
    except Exception as e:
        return bid, False, str(e)


tasks = [(int(row["bldg_id"]), row["in.state"]) for _, row in buildings_full.iterrows()]

ok, failed = 0, []
with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
    futures = {executor.submit(download_one, bid, state): (bid, state) for bid, state in tasks}
    for i, future in enumerate(as_completed(futures)):
        bid, success, error = future.result()
        if success:
            ok += 1
        else:
            state = futures[future][1]
            failed.append((bid, state, error))
        if (i + 1) % 200 == 0:
            print(f"  {i + 1:,}/{len(tasks):,} traités ({ok:,} ok, {len(failed):,} échecs)")

print(f"Séries disponibles : {ok:,} / {len(tasks):,}")
if failed:
    failed_df = pd.DataFrame(failed, columns=["bldg_id", "state", "error"])
    failed_df.to_parquet(DATA_PROCESSED / "buildings_full_download_failed.parquet", index=False)
    print(f"Échecs ({len(failed)}) sauvegardés dans buildings_full_download_failed.parquet")

## Bâtiments effectivement disponibles localement

Le pipeline repart de ce qui est réellement sur disque plutôt que de la liste de candidats : si le
téléchargement a été interrompu, ce notebook fonctionne quand même sur le sous-ensemble déjà
présent (re-tourner la cellule de téléchargement ci-dessus complète progressivement l'échantillon).

In [ ]:
parquet_files = sorted(DATA_RAW_FULL.glob("*-0.parquet"))
bldg_ids = [int(f.stem.split("-")[0]) for f in parquet_files]
print(f"Bâtiments disponibles localement : {len(bldg_ids):,} / {len(buildings_full):,} candidats")

## Features statiques : agrégats physiques de Boubaker + reste des features physiques

Même construction que dans `rnn_mlp_hybrid_electricite.ipynb` : les **5 agrégats physiques**
(`UA`, `H_ve`, `C`, `A_solaire`, `compacité`, définis dans `lgbm_electricity_5features.ipynb`)
remplacent les 4 indices d'enveloppe redondants (`thermal_envelope_index`, `window_index`,
`solar_gain_index`, `thermal_exposure`) de `X_physical_engineered.parquet`. Les agrégats sont
déjà précalculés pour les 549 971 logements du dataset complet dans
`data/processed/X_aggregates.parquet` (la fonction `build_aggregates` qui les a produits est
documentée dans `rnn_mlp_hybrid_electricite.ipynb`, pas reprise ici puisqu'elle n'a pas besoin
d'être réexécutée).

In [ ]:
meta = pd.read_parquet(DATA_PROCESSED / "metadata_clean.parquet", columns=["bldg_id"])
pos = meta.reset_index(drop=False).set_index("bldg_id").loc[bldg_ids, "index"].values

ENVELOPE_IDX = ["in.thermal_envelope_index", "in.window_index", "in.solar_gain_index", "in.thermal_exposure"]
X_physical = pd.read_parquet(DATA_PROCESSED / "X_physical_engineered.parquet")
non_envelope = X_physical.drop(columns=ENVELOPE_IDX).iloc[pos].reset_index(drop=True)

agg_all = pd.read_parquet(DATA_PROCESSED / "X_aggregates.parquet").set_index("bldg_id")
boubaker_agg = agg_all.loc[bldg_ids].reset_index(drop=True)

static_df = pd.concat([non_envelope, boubaker_agg], axis=1)
static_all = static_df.astype("float32").values
N_STATIC_FEATURES = static_all.shape[1]

print(f"Features statiques : {static_all.shape} "
      f"({non_envelope.shape[1]} non-enveloppe + {boubaker_agg.shape[1]} agrégats de Boubaker)")

## Chargement des séries horaires + journalières

À 100 bâtiments, toutes les séries tenaient facilement dans une liste puis un `np.stack`. À
l'échelle de la population complète, on préalloue directement le tableau final
(`hourly_all`, ~2 Go pour 6 700 bâtiments) et on le remplit en streaming, un fichier à la fois,
pour ne jamais garder deux copies en mémoire. Les rares bâtiments avec une année incomplète (trous
dans les données brutes) sont écartés plutôt que de faire échouer tout le chargement — à 100
bâtiments un `assert` suffisait, à plusieurs milliers ce n'est plus réaliste.

In [ ]:
agg = {COL: "sum", **{c: "mean" for c in WEATHER_COLS}}

hourly_all = np.full((len(parquet_files), N_HOURS, N_SEQ_FEATURES), np.nan, dtype="float32")
valid_bldg = np.zeros(len(parquet_files), dtype=bool)

for i, f in enumerate(parquet_files):
    d = pd.read_parquet(f, columns=["timestamp"] + SEQ_COLS)
    d = d.assign(timestamp=lambda x: pd.to_datetime(x["timestamp"]) - pd.Timedelta("15m")).set_index("timestamp")
    d_h = d[SEQ_COLS].resample("h").agg(agg)
    if len(d_h) != N_HOURS:
        continue  # année incomplète (rare) -- bâtiment écarté plutôt que de tout faire échouer
    hourly_all[i] = d_h[SEQ_COLS].values.astype("float32")
    valid_bldg[i] = True
    if (i + 1) % 500 == 0:
        print(f"  {i + 1:,}/{len(parquet_files):,} bâtiments chargés")

n_dropped = int((~valid_bldg).sum())
if n_dropped:
    print(f"{n_dropped} bâtiment(s) écarté(s) pour année incomplète")

hourly_all = hourly_all[valid_bldg]
bldg_ids = list(np.array(bldg_ids)[valid_bldg])
static_all = static_all[valid_bldg]
N_BLDG = len(bldg_ids)
print("Séries horaires :", hourly_all.shape)

# Séries journalières dérivées (consommation sommée, météo moyennée par jour) — réutilisées
# pour la Partie B, sans relire les fichiers bruts
hourly_reshaped = hourly_all.reshape(N_BLDG, 365, 24, N_SEQ_FEATURES)
daily_all = np.empty((N_BLDG, 365, N_SEQ_FEATURES), dtype="float32")
daily_all[:, :, 0] = hourly_reshaped[:, :, :, 0].sum(axis=2)      # consommation : somme journalière
daily_all[:, :, 1:] = hourly_reshaped[:, :, :, 1:].mean(axis=2)   # météo : moyenne journalière
print("Séries journalières :", daily_all.shape)

## Split train/val/test — au niveau bâtiment

Même logique que dans `rnn_mlp_hybrid_electricite.ipynb` (70 / 15 / 15, split par bâtiment plutôt
que par fenêtre temporelle, pour tester la généralisation à des bâtiments jamais vus) — mais
appliquée à toute la population disponible, donc plusieurs milliers de bâtiments dans chaque split
au lieu de 70/15/15.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
perm = rng.permutation(N_BLDG)
n_train = int(0.70 * N_BLDG)
n_val = int(0.15 * N_BLDG)
train_idx, val_idx, test_idx = perm[:n_train], perm[n_train:n_train + n_val], perm[n_train + n_val:]

print(f"Train : {len(train_idx):,} bâtiments")
print(f"Val   : {len(val_idx):,} bâtiments")
print(f"Test  : {len(test_idx):,} bâtiments")

## Architecture hybride commune (identique à `rnn_mlp_hybrid_electricite.ipynb`)

`rnn_type` est paramétrable (`"SimpleRNN"`, `"LSTM"`, `"GRU"`) — GRU par défaut. Architecture
"two-tower" / late fusion inchangée : une branche séquentielle (RNN/LSTM/GRU) et une branche
statique (MLP) sont encodées séparément puis concaténées avant la tête de régression commune.

In [ ]:
def build_hybrid_model(seq_len, n_seq_features, n_static_features,
                        rnn_type="GRU", rnn_units=32, static_units=(32, 16),
                        dropout=0.0):
    """Modèle hybride : branche séquentielle (RNN/LSTM/GRU) + branche statique (MLP),
    concaténées puis passées dans une tête de régression commune."""
    rnn_layer = {"SimpleRNN": layers.SimpleRNN, "LSTM": layers.LSTM, "GRU": layers.GRU}[rnn_type]

    seq_input = layers.Input(shape=(seq_len, n_seq_features), name="seq_input")
    x = rnn_layer(rnn_units)(seq_input)
    if dropout > 0:
        x = layers.Dropout(dropout)(x)

    static_input = layers.Input(shape=(n_static_features,), name="static_input")
    s = static_input
    for units in static_units:
        s = layers.Dense(units, activation="relu")(s)
    if dropout > 0:
        s = layers.Dropout(dropout)(s)

    combined = layers.Concatenate()([x, s])
    combined = layers.Dense(max(8, rnn_units // 2), activation="relu")(combined)
    output = layers.Dense(1, name="output")(combined)

    return keras.Model(inputs=[seq_input, static_input], outputs=output, name=f"hybrid_{rnn_type}")


def metrics(y_true, y_pred, label):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {"Modèle": label, "RMSE": rmse, "MAE": mae, "R2": r2}


def plot_metrics_bar(df_metrics, title, filename):
    """Barres comparatives RMSE / MAE, couleurs + hatching adaptés daltonisme (encodage
    redondant : la catégorie ne dépend jamais de la seule couleur)."""
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, metric in zip(axes, ["RMSE", "MAE"]):
        bars = ax.bar(df_metrics.index, df_metrics[metric],
                       color=[CB_PALETTE[i % len(CB_PALETTE)] for i in range(len(df_metrics))])
        for i, b in enumerate(bars):
            b.set_hatch(CB_HATCHES[i % len(CB_HATCHES)])
        ax.set_title(metric)
        ax.set_ylabel(metric)
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=20)
    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(FIGURES / filename, dpi=150, bbox_inches="tight")
    plt.show()

## Partie A — Prévision à court terme (t+1, fenêtre de 24h)

Fenêtre glissante de 24 heures (consommation + météo) → consommation de l'heure suivante. Avec
6 700+ bâtiments, l'ensemble des fenêtres possibles dépasse 40 millions rien que pour le train
— hors de portée de la RAM disponible si on les matérialise comme dans le notebook à 100
bâtiments. `RandomWindowSequence` ci-dessous construit chaque batch **à la volée** par indexation
directe dans `hourly_all` (tenu en RAM, non fenêtré), sans jamais matérialiser l'ensemble des
fenêtres :

- **train** : un sous-échantillon aléatoire de `N_WINDOWS_TRAIN_EPOCH` fenêtres, retiré à chaque
  epoch (`resample_each_epoch=True`) — sur plusieurs epochs, le modèle voit un large éventail de
  bâtiments et de moments de l'année sans jamais charger plus qu'un batch à la fois.
- **val/test** : un sous-échantillon aléatoire fixe de `N_WINDOWS_EVAL` fenêtres (mêmes fenêtres
  d'une évaluation à l'autre), pour des métriques reproductibles.

In [ ]:
LOOKBACK = 24
BATCH_SIZE = 512
N_WINDOWS_TRAIN_EPOCH = 500_000  # sous-échantillon aléatoire, renouvelé à chaque epoch
N_WINDOWS_EVAL = 300_000         # sous-échantillon fixe pour validation/test


def sample_window_index(bldg_pool, n_windows, rng, hourly_len=N_HOURS, lookback=LOOKBACK):
    """Tire n_windows paires (indice bâtiment, heure de fin de fenêtre) parmi bldg_pool.
    La fenêtre associée à (b, t) est hourly_all[b, t-lookback:t], la cible hourly_all[b, t, 0]."""
    b = rng.choice(bldg_pool, size=n_windows, replace=True).astype("int32")
    t = rng.integers(lookback, hourly_len, size=n_windows).astype("int32")
    return b, t


class RandomWindowSequence(keras.utils.Sequence):
    """Assemble les batches (fenêtre séquentielle, features statiques, cible) à la volée à
    partir de `hourly_all` tenu en RAM, sans matérialiser l'ensemble des fenêtres (impossible à
    cette échelle, cf. note ci-dessus). Un sous-ensemble aléatoire de `n_windows` fenêtres est
    tiré parmi les bâtiments de `bldg_pool` ; renouvelé à chaque epoch si
    `resample_each_epoch=True` (train), fixe sinon (val/test)."""

    def __init__(self, hourly_all, static_table, bldg_pool, n_windows, batch_size,
                 seq_scaler, y_scaler, resample_each_epoch=False, seed=RANDOM_STATE):
        self.hourly_all = hourly_all
        self.static_table = static_table
        self.bldg_pool = np.asarray(bldg_pool)
        self.n_windows = n_windows
        self.batch_size = batch_size
        self.seq_scaler = seq_scaler
        self.y_scaler = y_scaler
        self.resample_each_epoch = resample_each_epoch
        self.rng = np.random.default_rng(seed)
        self._draw()

    def _draw(self):
        self.b_idx, self.t_idx = sample_window_index(self.bldg_pool, self.n_windows, self.rng)

    def __len__(self):
        return int(np.ceil(self.n_windows / self.batch_size))

    def __getitem__(self, i):
        sl = slice(i * self.batch_size, (i + 1) * self.batch_size)
        b, t = self.b_idx[sl], self.t_idx[sl]

        X = np.stack([self.hourly_all[bi, ti - LOOKBACK:ti] for bi, ti in zip(b, t)]).astype("float32")
        y = self.hourly_all[b, t, 0].astype("float32")
        S = self.static_table[b]

        X = self.seq_scaler.transform(X.reshape(-1, X.shape[-1])).reshape(X.shape).astype("float32")
        y = self.y_scaler.transform(y.reshape(-1, 1)).astype("float32").ravel()

        return {"seq_input": X, "static_input": S}, y

    def on_epoch_end(self):
        if self.resample_each_epoch:
            self._draw()


# Scalers fités sur un sous-échantillon de fenêtres tirées UNIQUEMENT parmi les bâtiments de
# train (jamais la matérialisation complète des 40M+ fenêtres possibles).
rng_fit = np.random.default_rng(RANDOM_STATE)
b_fit, t_fit = sample_window_index(train_idx, 200_000, rng_fit)
X_fit = np.stack([hourly_all[bi, ti - LOOKBACK:ti] for bi, ti in zip(b_fit, t_fit)]).astype("float32")
y_fit = hourly_all[b_fit, t_fit, 0].astype("float32")

seq_scaler_a = StandardScaler().fit(X_fit.reshape(-1, N_SEQ_FEATURES))
y_scaler_a = StandardScaler().fit(y_fit.reshape(-1, 1))
del X_fit, y_fit

train_seq_a = RandomWindowSequence(hourly_all, static_all, train_idx, N_WINDOWS_TRAIN_EPOCH, BATCH_SIZE,
                                    seq_scaler_a, y_scaler_a, resample_each_epoch=True)
val_seq_a = RandomWindowSequence(hourly_all, static_all, val_idx, N_WINDOWS_EVAL, BATCH_SIZE,
                                  seq_scaler_a, y_scaler_a, resample_each_epoch=False)

print(f"Fenêtres/epoch (train) : {N_WINDOWS_TRAIN_EPOCH:,}  |  Fenêtres val (fixes) : {N_WINDOWS_EVAL:,}")

In [ ]:
model_a = build_hybrid_model(LOOKBACK, N_SEQ_FEATURES, N_STATIC_FEATURES, rnn_type="GRU", rnn_units=32)
model_a.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse", metrics=["mae"])
model_a.summary()

early_stop_a = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

history_a = model_a.fit(
    train_seq_a,
    validation_data=val_seq_a,
    epochs=15, callbacks=[early_stop_a], verbose=1,
)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history_a.history["loss"], label="train", color=CB_PALETTE[0])
plt.plot(history_a.history["val_loss"], label="val", color=CB_PALETTE[1], linestyle="--")
plt.title("Partie A — Courbe d'apprentissage (MSE, cible standardisée, population complète)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES / "rnn_mlp_full_loss_modele_a_previsionhoraire.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
test_seq_a = RandomWindowSequence(hourly_all, static_all, test_idx, N_WINDOWS_EVAL, BATCH_SIZE,
                                   seq_scaler_a, y_scaler_a, resample_each_epoch=False)
pred_test_a_sc = model_a.predict(test_seq_a, verbose=0)
pred_test_a = y_scaler_a.inverse_transform(pred_test_a_sc).ravel()

ya_test = hourly_all[test_seq_a.b_idx, test_seq_a.t_idx, 0]
persistence_pred = hourly_all[test_seq_a.b_idx, test_seq_a.t_idx - 1, 0]  # dernière heure observée (baseline)

df_metrics_a = pd.DataFrame([
    metrics(ya_test, persistence_pred, "Persistance (t = t-1)"),
    metrics(ya_test, pred_test_a, f"Hybride GRU+MLP ({LOOKBACK}h, {N_BLDG:,} bâtiments)"),
]).set_index("Modèle")

df_metrics_a.round(4)

In [ ]:
rng_plot = np.random.default_rng(RANDOM_STATE)
n_points = 3000
sample = rng_plot.choice(len(ya_test), size=min(n_points, len(ya_test)), replace=False)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(ya_test[sample], pred_test_a[sample], alpha=0.25, s=8, color=CB_PALETTE[0])
lim = [0, max(ya_test[sample].max(), pred_test_a[sample].max())]
ax.plot(lim, lim, color=CB_PALETTE[7], linestyle="--", linewidth=2)
ax.set_xlabel("Consommation réelle à t+1 (kWh)")
ax.set_ylabel("Consommation prédite à t+1 (kWh)")
ax.set_title(f"Partie A — Réel vs prédit (échantillon de {N_WINDOWS_EVAL:,} fenêtres de test, {len(test_idx):,} bâtiments)")
plt.tight_layout()
plt.savefig(FIGURES / "rnn_mlp_full_tacheA_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
plot_metrics_bar(
    df_metrics_a,
    "Partie A — Comparaison des métriques (persistance vs hybride, population complète)",
    "rnn_mlp_full_partieA_metrics_bar.png",
)

In [ ]:
residuals_a = pred_test_a - ya_test

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(residuals_a, bins=80, color=CB_PALETTE[0], alpha=0.8)
axes[0].axvline(0, color=CB_PALETTE[7], linestyle="--")
axes[0].set_xlabel("Résidu (prédit − réel, kWh)")
axes[0].set_ylabel("Nombre de fenêtres")
axes[0].set_title("Distribution des résidus")

axes[1].scatter(ya_test[sample], residuals_a[sample], alpha=0.2, s=8, color=CB_PALETTE[0])
axes[1].axhline(0, color=CB_PALETTE[7], linestyle="--")
axes[1].set_xlabel("Consommation réelle à t+1 (kWh)")
axes[1].set_ylabel("Résidu (prédit − réel, kWh)")
axes[1].set_title("Résidus vs valeur réelle")

fig.suptitle("Partie A — Analyse des résidus (population complète)")
plt.tight_layout()
plt.savefig(FIGURES / "rnn_mlp_full_partieA_residus.png", dpi=150, bbox_inches="tight")
plt.show()

### Zoom temporel sur un bâtiment de test

Contrairement au reste de la Partie A (fenêtres tirées aléatoirement), ce graphique reconstruit
**toutes** les fenêtres d'un seul bâtiment de test sur ses deux premières semaines — pour
visualiser la dynamique temporelle plutôt qu'un nuage de points indépendants.

In [ ]:
zoom_pos = test_idx[0]
t_range = np.arange(LOOKBACK, N_HOURS)

X_zoom = np.stack([hourly_all[zoom_pos, t - LOOKBACK:t] for t in t_range]).astype("float32")
X_zoom_sc = seq_scaler_a.transform(X_zoom.reshape(-1, N_SEQ_FEATURES)).reshape(X_zoom.shape).astype("float32")
S_zoom = np.repeat(static_all[zoom_pos][None, :], len(t_range), axis=0)

pred_zoom_sc = model_a.predict({"seq_input": X_zoom_sc, "static_input": S_zoom}, verbose=0)
pred_zoom = y_scaler_a.inverse_transform(pred_zoom_sc).ravel()
y_zoom = hourly_all[zoom_pos, t_range, 0]

n_show = 24 * 14  # deux semaines
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(np.arange(n_show), y_zoom[:n_show], label="Réel", color=CB_PALETTE[0], linewidth=1.2)
ax.plot(np.arange(n_show), pred_zoom[:n_show], label="Prédit", color=CB_PALETTE[1], linestyle="--", linewidth=1.2, alpha=0.9)
ax.set_xlabel("Heure")
ax.set_ylabel("Consommation (kWh)")
ax.set_title(f"Partie A — Réel vs prédit dans le temps (bâtiment test #{bldg_ids[zoom_pos]}, 2 premières semaines)")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "rnn_mlp_full_partieA_timeseries.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import gc

# Libère la RAM de la Partie A avant la Partie B (Xa_fit déjà supprimé plus haut ; ce qui reste
# de lourd est le modèle/graphe Keras).
del model_a, train_seq_a, val_seq_a, test_seq_a
keras.backend.clear_session()
gc.collect()

# clear_session() décale le flux aléatoire global de TF -> reseed pour que la Partie B soit
# reproductible indépendamment de ce qui s'est passé en Partie A
tf.random.set_seed(RANDOM_STATE)

## Partie B — Extrapolation du total annuel (90 premiers jours observés)

À partir des 90 premiers jours de l'année (consommation + météo, au pas **journalier**), on
extrapole la consommation électrique totale de l'année complète.

Contrairement au notebook à 100 bâtiments (70 exemples d'entraînement, réseau volontairement
bridé — peu d'unités, dropout fort — pour limiter le sur-apprentissage), l'échantillon
d'entraînement compte ici plusieurs milliers de bâtiments : c'est directement la piste
d'amélioration identifiée dans la synthèse de `rnn_mlp_hybrid_electricite.ipynb`. La capacité du
réseau est donc légèrement augmentée (mêmes tailles de couches que la Partie A) puisqu'elle n'a
plus besoin d'être bridée pour éviter le sur-apprentissage sur un échantillon aussi petit.

In [ ]:
K_DAYS = 90

X_seq_annual = daily_all[:, :K_DAYS, :]          # (N_BLDG, 90, n_seq_features)
y_annual = daily_all[:, :, 0].sum(axis=1)        # consommation totale sur les 365 jours

Xb_train, Xb_val, Xb_test = X_seq_annual[train_idx], X_seq_annual[val_idx], X_seq_annual[test_idx]
Sb_train, Sb_val, Sb_test = static_all[train_idx], static_all[val_idx], static_all[test_idx]
yb_train, yb_val, yb_test = y_annual[train_idx], y_annual[val_idx], y_annual[test_idx]

print(f"Train : {Xb_train.shape[0]:,} bâtiments  |  Val : {Xb_val.shape[0]:,}  |  Test : {Xb_test.shape[0]:,}")

In [ ]:
seq_scaler_b = StandardScaler().fit(Xb_train.reshape(-1, N_SEQ_FEATURES))
Xb_train_sc = seq_scaler_b.transform(Xb_train.reshape(-1, N_SEQ_FEATURES)).reshape(Xb_train.shape).astype("float32")
Xb_val_sc = seq_scaler_b.transform(Xb_val.reshape(-1, N_SEQ_FEATURES)).reshape(Xb_val.shape).astype("float32")
Xb_test_sc = seq_scaler_b.transform(Xb_test.reshape(-1, N_SEQ_FEATURES)).reshape(Xb_test.shape).astype("float32")

static_scaler_b = StandardScaler().fit(Sb_train)
Sb_train_sc = static_scaler_b.transform(Sb_train).astype("float32")
Sb_val_sc = static_scaler_b.transform(Sb_val).astype("float32")
Sb_test_sc = static_scaler_b.transform(Sb_test).astype("float32")

y_scaler_b = StandardScaler().fit(yb_train.reshape(-1, 1))
yb_train_sc = y_scaler_b.transform(yb_train.reshape(-1, 1)).astype("float32").ravel()
yb_val_sc = y_scaler_b.transform(yb_val.reshape(-1, 1)).astype("float32").ravel()

In [ ]:
model_b = build_hybrid_model(
    K_DAYS, N_SEQ_FEATURES, N_STATIC_FEATURES,
    rnn_type="GRU", rnn_units=32, static_units=(32, 16), dropout=0.2,
)
model_b.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse", metrics=["mae"])
model_b.summary()

early_stop_b = keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

history_b = model_b.fit(
    [Xb_train_sc, Sb_train_sc], yb_train_sc,
    validation_data=([Xb_val_sc, Sb_val_sc], yb_val_sc),
    epochs=100, batch_size=64, callbacks=[early_stop_b], verbose=1,
)
print(f"Arrêt à l'epoch {len(history_b.history['loss'])} (early stopping)")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history_b.history["loss"], label="train", color=CB_PALETTE[0])
plt.plot(history_b.history["val_loss"], label="val", color=CB_PALETTE[1], linestyle="--")
plt.title("Partie B — Courbe d'apprentissage (MSE, cible standardisée, population complète)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES / "rnn_mlp_full_loss_modele_b_extrapolationannuelle.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
pred_test_b_sc = model_b.predict([Xb_test_sc, Sb_test_sc], verbose=0)
pred_test_b = y_scaler_b.inverse_transform(pred_test_b_sc).ravel()

dummy_pred_b = np.full_like(yb_test, yb_train.mean())

df_metrics_b = pd.DataFrame([
    metrics(yb_test, dummy_pred_b, "Dummy (moyenne du train)"),
    metrics(yb_test, pred_test_b, f"Hybride GRU+MLP ({K_DAYS}j observés, {len(train_idx):,} bâtiments train)"),
]).set_index("Modèle")

df_metrics_b.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(yb_test, pred_test_b, s=20, alpha=0.4, color=CB_PALETTE[0])
lim = [0, max(yb_test.max(), pred_test_b.max())]
ax.plot(lim, lim, color=CB_PALETTE[7], linestyle="--", linewidth=2)
ax.set_xlabel("Consommation annuelle réelle (kWh)")
ax.set_ylabel("Consommation annuelle prédite (kWh)")
ax.set_title(f"Partie B — Réel vs prédit ({len(yb_test):,} bâtiments de test)")
plt.tight_layout()
plt.savefig(FIGURES / "rnn_mlp_full_tacheB_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
plot_metrics_bar(
    df_metrics_b,
    "Partie B — Comparaison des métriques (dummy vs hybride, population complète)",
    "rnn_mlp_full_partieB_metrics_bar.png",
)

### Réel vs prédit par bâtiment (échantillon illustratif)

Avec `len(test_idx)` bâtiments de test (plusieurs centaines à cette échelle, contre 15 dans le
notebook à 100 bâtiments), un graphique en barres avec une barre par bâtiment n'est plus lisible.
On affiche donc un sous-échantillon aléatoire de 50 bâtiments de test — le scatter plot ci-dessus
reste la référence pour l'ensemble du test set.

In [ ]:
rng_bar = np.random.default_rng(RANDOM_STATE)
n_bar = min(50, len(yb_test))
bar_sample = rng_bar.choice(len(yb_test), size=n_bar, replace=False)

order = bar_sample[np.argsort(yb_test[bar_sample])]
bldg_labels = [str(b) for b in np.array(bldg_ids)[test_idx][order]]

x = np.arange(n_bar)
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width / 2, yb_test[order], width, label="Réel", color=CB_PALETTE[0])
ax.bar(x + width / 2, pred_test_b[order], width, label="Prédit", color=CB_PALETTE[1], hatch="//")
ax.set_xticks(x)
ax.set_xticklabels(bldg_labels, rotation=45, ha="right", fontsize=7)
ax.set_xlabel("Bâtiment (id)")
ax.set_ylabel("Consommation annuelle (kWh)")
ax.set_title(f"Partie B — Réel vs prédit, échantillon de {n_bar} bâtiments de test (triés par consommation réelle)")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "rnn_mlp_full_partieB_bar.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print("Synthèse — chiffres réels de ce run")
print(f"Bâtiments chargés : {N_BLDG:,} (sur {len(candidates):,} candidats identifiés par le filtre)")
print(f"Split : {len(train_idx):,} train / {len(val_idx):,} val / {len(test_idx):,} test")

## Synthèse

- **Population complète** : le nombre de bâtiments effectivement chargés (cellule ci-dessus) et
  le split train/val/test qui en découle remplacent les 100 bâtiments (70/15/15) de
  `rnn_mlp_hybrid_electricite.ipynb`.
- **Partie A** : même architecture hybride (GRU + MLP) qu'à 100 bâtiments, mais entraînée sur un
  sous-échantillon aléatoire de fenêtres renouvelé à chaque epoch (`RandomWindowSequence`) plutôt
  que sur l'ensemble matérialisé des fenêtres — seule façon de tenir dans la RAM disponible à
  cette échelle. À comparer aux métriques de `rnn_mlp_hybrid_electricite.ipynb` (`df_metrics_a`
  des deux notebooks) pour juger de l'apport, ou non, de la diversité de bâtiments supplémentaire
  par rapport à l'échantillon à 100.
- **Partie B** : c'est ici que le gain attendu est le plus net — l'échantillon d'entraînement
  passe de 70 à plusieurs milliers de bâtiments, ce qui règle directement la limite principale du
  notebook à 100 bâtiments (réseau sous-paramétré par rapport au nombre d'exemples, métriques de
  test à forte variance sur seulement 15 bâtiments). Les métriques de test (`df_metrics_b`)
  portent maintenant sur un ensemble de test bien plus large, nettement plus représentatif.
- **Limite restante** : le sous-échantillonnage aléatoire des fenêtres en Partie A
  (`N_WINDOWS_TRAIN_EPOCH`/`N_WINDOWS_EVAL`) reste un choix d'ingénierie pour tenir dans 12 Go de
  RAM, pas un choix de modélisation — augmenter ces tailles (ou passer à un pipeline `tf.data`
  lisant directement depuis les fichiers parquet sur disque) serait la prochaine étape si les
  résultats de la Partie A doivent être encore consolidés.